# Texture Reduction Example 

This notebook demonstrates a prototype of texture data reduction workflow for POWTEX using a simulated dataset. 

## Imports

In [ ]:
from mlz.diffraction.texture.types import *
from mlz.diffraction.texture.workflow import TextureReduce

import matplotlib.pyplot as plt
import plopp as pp
import numpy as np

%matplotlib widget

## Configure workflow

The workflow diagram below summarizes the main reduction steps, ordered from left to right. Red boxes highlight input parameters that are still missing and must be specified before the workflow can run. Some input parameters (e.g. DperpBins, DspacingBins) have default values that can be overridden.

In [ ]:
texture_workflow = TextureReduce()
texture_workflow.visualize(graph_attr={"rankdir": "LR"})

In [ ]:
# TODO: files to be uploaded to the Scipp data server and loaded from there instead of local path
project_folder = "/home/koshchii/Instruments/POWTEX/Scipp/data/texture/csv"

# Set locations of measurement data
texture_workflow[Filepath[CalibrationRun]] = project_folder
texture_workflow[Filepath[SampleRun]] = project_folder
texture_workflow[Filepath[EmptyCanRun]] = project_folder
texture_workflow[Filepath[VanadiumRun]] = project_folder
texture_workflow[Filepath[EmptyVanadiumCanRun]] = project_folder


# Set measurement filename(s)
sample_filename = "data_powtex_biosample.csv"
dimond_filename = None
vanadium_filename = None
empty_filename = None
empty_vanadium_filename = None

texture_workflow[Filename[SampleRun]] = sample_filename
# This is hard coded default value for workflow to work in its current design
texture_workflow[Filename[CalibrationRun]] = dimond_filename
texture_workflow[Filename[VanadiumRun]] = vanadium_filename
texture_workflow[Filename[EmptyCanRun]] = empty_filename
texture_workflow[Filename[EmptyVanadiumCanRun]] = empty_vanadium_filename

# Set histogramming setting

texture_workflow[TwoThetaBinWidth] = sc.scalar(0.5, unit="deg")
texture_workflow[GammaBinWidth] = sc.scalar(1.0, unit="deg")
texture_workflow[DspacingBinWidth] = sc.scalar(0.02, unit="angstrom")

texture_workflow.visualize(graph_attr={"rankdir": "LR"})

### 2D flat view

Below, we take a look at the raw sample 2d data projected on the detector surface.

In [ ]:
flat_view_raw_data = texture_workflow.compute(tp=FlatViewRawData)
flat_view_raw_data.plot()

### $2\theta$-rings view

To visualise $2\theta$ rings in polar coordinates using plopp slicer, we need to sligtly modify the output of two_theta_rings_data by adding an additional dimension "r" and converting gamma values to radians

In [ ]:
# Prepare data for plotting in polar coordinates
two_theta_rings_data = texture_workflow.compute(tp=FlatViewRawData)
two_theta_rings_data = sc.flatten(two_theta_rings_data, dims=[], to="r")
two_theta_rings_data = sc.concat([two_theta_rings_data, two_theta_rings_data], dim="r")
two_theta_rings_data.coords["r"] = sc.Variable(values=[700, 750, 800], dims=["r"], unit="mm")
two_theta_rings_data = sc.transpose(two_theta_rings_data, dims=["two_theta", "r", "gamma"])
two_theta_rings_data.coords["gamma"] = two_theta_rings_data.coords["gamma"].to(unit="rad")

In [ ]:
plt.ioff()
fig, ax = plt.subplots(subplot_kw={"projection": "polar"})
ax.set_rorigin(0)
ax.set_yticks([])
ax.set_yticklabels([])

rings = pp.slicer(
    two_theta_rings_data,
    grid=True,
    ax=ax,
    mode="single",
    autoscale=True,
    clabel="Counts",
    title="2θ rings"
)

ax.set_ylabel("")
ax.set_xlabel("$\\gamma$ [deg]")

# making hower to show gamma in degrees and remove r values, as they may be misleading in this plot
ax.format_coord = lambda theta, r, fmt=ax.format_coord: f"γ: {np.rad2deg(theta):.2f}°" + fmt(theta, r).partition(")")[2]

display(rings)


## $\gamma$ slices view

An alternative way to look at the data is via a 1D $\gamma$ slicer view

In [ ]:
gamma_slices_view_raw_data = texture_workflow.compute(tp=GammaSlicesViewRawData)
fig = pp.slicer(gamma_slices_view_raw_data, keep="gamma", mode="single", linestyle="solid", marker=None)
# set the first slice to be displayed by default
fig.bottom_bar.children[0].controls["two_theta"].value = 0
fig

## Dspacing and $2\theta$ histograms of the reduced data

In [ ]:
dspacing_data = texture_workflow.compute(tp=DspacingViewData)
fig = pp.slicer(dspacing_data, mode="single")
# set the first slice to be displayed by default
fig.bottom_bar.children[0].controls["gamma"].value = 0
fig

In [ ]:
two_theta_data = texture_workflow.compute(tp=TwoThetaViewData)
fig = pp.slicer(two_theta_data, mode="single")
# set the first slice to be displayed by default
fig.bottom_bar.children[0].controls["gamma"].value = 0
fig

In [ ]:
integrated_two_theta_data = texture_workflow.compute(tp=IntegratedTwoThetaViewData)
integrated_two_theta_data.plot()
